In [4]:
!pip install trl transformers peft accelerate bitsandbytes

In [6]:
import os
import re
import gc
import json
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from collections import defaultdict

import torch
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel



# ============================================================
# 1. Phase 3 配置
# ============================================================
@dataclass
class Phase2Config:
    # 数据集
    dataset_name: str          = "tomg-group-umd/cinepile"
    train_split: str           = "train"
    test_split: str            = "test"

    # 分层抽样（None = 用完整 split）
    max_train_samples: Optional[int] = 30000   # e.g. 从 ~30 万中抽 3 万
    max_test_samples: Optional[int]  = None     # 测试集可截断或用 full

    # hard 样本在分层抽样中的额外倍率（>1 表示更偏向抽 hard）
    hard_oversample_ratio: float     = 1.5   ### Increase hard sample oversampling, training needs stronger representation

    # 模型
    base_model_id: str         = "meta-llama/Llama-3.1-8B-Instruct"

    # 输入长度
    max_scene_length: int      = 1200   ### Change from 1000 to 1200 (allow more scene context)
    max_seq_length: int        = 1536   ### Change from 1024 to 1536 (better reasoning window)

    # 通用训练参数
    num_train_epochs: int      = 4      ### Change from 4 to 5 (LoRA rarely overfits, improves learning)
    per_device_train_batch_size: int = 4   ### Change from 4 to 1 (T4 VRAM stability)
    gradient_accumulation_steps: int = 4  ### Change from 4 to 16 (keep effective batch size \u224816)
    learning_rate: float       = 2e-4
    warmup_ratio: float        = 0.05
    lr_scheduler_type: str     = "cosine"

    bf16: bool                 = True   ### Change to True for A100
    fp16: bool                 = False

    logging_steps: int         = 50

    # checkpoint & 输出
    save_steps: int            = 200
    save_total_limit: int      = 3
    output_base_dir: str       = "./phase2_outputs"
    output_csv: str            = "phase2_results.csv"

    # QLoRA
    lora_r: int                = 64 ### Consider a higher value if the model is struggling to capture the complexity
    lora_alpha: int            = 128
    lora_dropout: float        = 0.00 ### Since we have big example sizes (30000), maybe try 0
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]) ### Recommend all 7 layers, but might also try less

    # Prefix Tuning
    prefix_num_virtual_tokens: int = 20

    # 评估生成长度
    max_new_tokens: int        = 4

    # 类别映射
    category_map: Dict[str, str] = field(default_factory=lambda: {
        "TEMP": "Temporal",
        "CRD":  "Character and",
        "NPA":  "Narrative and",
        "STA":  "Setting and",
        "TH":   "Theme Exploration",
    })


@dataclass
class Phase3Config(Phase2Config):
    # 重用 Phase2Config 里的参数，额外加 Phase3 的一些路径
    phase3_output_dir: str = "./phase3_outputs"
    phase3_pred_file_base: str = "phase3_predictions_base.jsonl"
    phase3_pred_file_qlora: str = "phase3_predictions_qlora.jsonl"


# ============================================================
# 数据加载 & 预处理
# ============================================================
class CinePileDataset:
    LETTERS = ["A", "B", "C", "D", "E"]

    def __init__(self, cfg: Phase2Config):
        self.cfg = cfg
        self.test_data = self._load_all()
        print(f"[Dataset] Test: {len(self.test_data)}")

    @classmethod
    def _answer_text_to_letter(cls, answer_text: str, choices: list) -> str:
        for i, c in enumerate(choices):
            if answer_text.strip() == c.strip():
                return cls.LETTERS[i]
        for i, c in enumerate(choices):
            if answer_text.strip() in c or c.strip() in answer_text:
                return cls.LETTERS[i]
        return "A"

    @staticmethod
    def _normalize_hard_split(raw) -> bool:
        if isinstance(raw, bool):
            return raw
        return str(raw).strip().lower() == "true"

    def _load_split(self, split: str) -> list:
        raw = load_dataset(self.cfg.dataset_name, split=split)
        samples = []
        for ex in raw:
            letter = self._answer_text_to_letter(ex["answer_key"], ex["choices"])
            samples.append({
                "movie_scene":       ex["movie_scene"],
                "question":          ex["question"],
                "choices":           ex["choices"],
                "answer_key":        letter,
                "question_category": ex["question_category"],
                "hard_split":        self._normalize_hard_split(ex["hard_split"]),
            })
        return samples

    def _load_all(self):
        # train_raw = self._load_split(self.cfg.train_split)
        test_raw  = self._load_split(self.cfg.test_split)

        # 训练集分层抽样
        # sampler    = StratifiedSampler(self.cfg)
        # train_data = sampler.sample(train_raw)

        # 测试集截断
        if self.cfg.max_test_samples:
            test_data = test_raw[:self.cfg.max_test_samples]
        else:
            test_data = test_raw

        # return train_data, test_data

        return test_data


# ============================================================
# 2. Prompt 风格与 paraphrase
#    Base style MUST match Phase2 training prompt exactly
# ============================================================
class Phase3PromptBuilder:
    """
    Base 风格必须与 Phase 2 训练 / 推理的 PromptBuilder.build_inference_prompt 完全一致。
    下面直接复制你原来的实现。
    """

    @staticmethod
    def build_base_prompt(sample: dict, max_scene_length: int) -> str:
        # === EXACT COPY of Phase2 PromptBuilder.build_inference_prompt ===
        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )
        return (
            "You are watching a movie. Based on the scene description, "
            "answer the multiple choice question.\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            "Think briefly about the scene before choosing the answer.\n\n"
            f"Options:\n{options}\n\n"
            "Answer (A/B/C/D/E):"
        )

    @staticmethod
    def build_prompt(style_name: str, style_prompt: str, sample: dict, max_scene_length: int) -> str:
        """
        style_prompt 是一段“指令部分”（不含 Scene/Question/Options），
        我们把它和 CinePile 内容拼接成完整 prompt。
        对 Base 风格，style_prompt 可以忽略，直接调用 build_base_prompt。
        """
        if style_name == "base":
            # 保证与 Phase 2 完全一致
            return Phase3PromptBuilder.build_base_prompt(sample, max_scene_length)

        scene   = sample["movie_scene"][:max_scene_length]
        options = "\n".join(
            f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])
        )

        # 这里 Strict / CoT 风格保持和 Base 结构类似，只替换指令段
        prompt = (
            f"{style_prompt}\n\n"
            f"Scene: {scene}\n\n"
            f"Question: {sample['question']}\n\n"
            "Think briefly about the scene before choosing the answer.\n\n"
            f"Options:\n{options}\n\n"
            "Answer (A/B/C/D/E):"
        )
        return prompt


# 三种风格 + 每种 3 个 paraphrase
PROMPT_STYLES: Dict[str, List[str]] = {

    "base": [
        "You are watching a movie. Based on the scene description, answer the multiple-choice question.",
        "Answer the following multiple-choice question about the movie after reviewing the provided scene description.",
        "Given the details of this movie scene, select the correct option for the question provided below."
    ],
    "strict": [
        "You are watching a movie. Based on the scene description, answer the multiple-choice question. Respond with only the letter of the correct option (A, B, C, D, or E).",
        "Examine the movie scene and choose the right answer. Your response must contain nothing but the single capital letter (A-E) representing your choice.",
        "Pick the best option (A, B, C, D, or E) for the movie question below based on the scene. Provide only the letter as your output."
    ],
    "cot": [
        "You are watching a movie. Based on the scene description, think step by step about the question and options, and then answer with the letter of the correct option.",
        "Read the movie scene carefully. Analyze the question and each possible choice logically before providing the final answer letter (A-E).",
        "Evaluate the provided movie scene. First, explain your reasoning process for selecting the best answer, then conclude with the correct letter (A-E)."
    ]

}

CHOICE_LETTERS = ["A", "B", "C", "D", "E"]


# ============================================================
# 3. 答案解析
# ============================================================
def extract_choice_from_output(text: str) -> str:
    """
    与 Phase2 Evaluator._predict 中逻辑保持一致：寻找 A-E。
    """
    decoded = text.strip().upper()
    match = re.search(r"\b([A-E])\b", decoded)
    if match:
        return match.group(1)
    return None  # 解析失败，后续可以单独统计


# ============================================================
# 4. 通用推理函数：单模型 \u00d7 多 prompt
# ============================================================
@torch.inference_mode()
def run_phase3_inference_for_model(
    model,
    tokenizer,
    cfg: Phase3Config,
    test_data: List[dict],
    model_name: str,
    output_path: str,
    max_new_tokens: int = 4,
    batch_size: int = 4,
):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    model.eval()

    all_records = []

    for style_name, paraphrases in PROMPT_STYLES.items():
        for pid, style_prompt in enumerate(paraphrases):
            print(f"[Phase3] Model={model_name} | style={style_name} | paraphrase={pid}")
            for idx in tqdm(range(0, len(test_data)), desc=f"{style_name}-{pid}"):
                sample = test_data[idx]
                prompt = Phase3PromptBuilder.build_prompt(
                    style_name=style_name,
                    style_prompt=style_prompt,
                    sample=sample,
                    max_scene_length=cfg.max_scene_length,
                )

                inputs = tokenizer(
                    prompt,
                    return_tensors="pt",
                    truncation=True,
                    max_length=cfg.max_seq_length,
                )
                inputs = {k: v.to(model.device) for k, v in inputs.items()}

                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                )

                generated = outputs[0][inputs["input_ids"].shape[1]:]
                text = tokenizer.decode(generated, skip_special_tokens=True)
                pred_choice = extract_choice_from_output(text)

                all_records.append({
                    "model": model_name,
                    "index": idx,
                    "style": style_name,
                    "paraphrase_id": pid,
                    "pred": pred_choice,
                    "label": sample["answer_key"],
                    "question_category": sample["question_category"],
                    "hard_split": sample["hard_split"],
                })

    with open(output_path, "w", encoding="utf-8") as f:
        for r in all_records:
            f.write(json.dumps(r) + "\n")

    print(f"[Phase3] Saved predictions \u2192 {output_path}")
    return all_records


# ============================================================
# 5. Phase 3 Runner
# ============================================================
class Phase3Runner:
    def __init__(self, cfg: Phase3Config):
        self.cfg = cfg
        self.dataset = CinePileDataset(cfg)  # 复用 Phase2 的加载 + stratified sampling

        # Updated print statement as train_data is no longer loaded
        print(f"[Phase3] Loaded dataset - Test: {len(self.dataset.test_data)}")

    def load_base_model(self):
        print("[Phase3] Loading base model...")
        base_model = AutoModelForCausalLM.from_pretrained(
            self.cfg.base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        tokenizer = AutoTokenizer.from_pretrained(self.cfg.base_model_id)
        return base_model, tokenizer

    # Changed to an instance method and arguments updated for correct passing
    def load_qlora_merged_model(self, base_model_id: str, adapter_path:str):

        print("[Merge] Loading base model...")

        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

        tokenizer = AutoTokenizer.from_pretrained(base_model_id)

        print("[Merge] Loading LoRA adapter...")
        model = PeftModel.from_pretrained(base_model, adapter_path)

        print("[Merge] Merging LoRA weights...")
        model = model.merge_and_unload()

        model.eval()

        return model, tokenizer

    # def load_qlora_merged_model(self, adapter_dir: str):
    #     """
    #     直接复用你 Phase2 的 merge 函数。
    #     adapter_dir 应该指向 phase2 qlora_rXX 的输出目录。
    #     """
    #     model, tokenizer = load_merged_model(self.cfg.base_model_id, adapter_dir)
    #     return model, tokenizer

    def run(self, qlora_adapter_dir: str):
        os.makedirs(self.cfg.phase3_output_dir, exist_ok=True)

        # 1. Base model
        base_model, base_tok = self.load_base_model()
        base_out_path = os.path.join(self.cfg.phase3_output_dir, self.cfg.phase3_pred_file_base)
        run_phase3_inference_for_model(
            model=base_model,
            tokenizer=base_tok,
            cfg=self.cfg,
            test_data=self.dataset.test_data,
            model_name="base",
            output_path=base_out_path,
            max_new_tokens=self.cfg.max_new_tokens,
        )
        del base_model, base_tok
        gc.collect()
        torch.cuda.empty_cache()

        # 2. QLoRA merged model
        # Corrected call to load_qlora_merged_model
        qlora_model, qlora_tok = self.load_qlora_merged_model(self.cfg.base_model_id, qlora_adapter_dir)
        qlora_out_path = os.path.join(self.cfg.phase3_output_dir, self.cfg.phase3_pred_file_qlora)
        run_phase3_inference_for_model(
            model=qlora_model,
            tokenizer=qlora_tok,
            cfg=self.cfg,
            test_data=self.dataset.test_data,
            model_name="qlora",
            output_path=qlora_out_path,
            max_new_tokens=self.cfg.max_new_tokens,
        )
        del qlora_model, qlora_tok
        gc.collect()
        torch.cuda.empty_cache()

        print("[Phase3] Inference done. You can now run separate analysis scripts "
              "to compute accuracy, variance, and per-question sensitivity.")


# ============================================================
# 6. main
# ============================================================
if __name__ == "__main__":

    from google.colab import drive
    drive.mount('/content/drive')

    PROJECT_DIR = "/content/drive/MyDrive/266_final/phase3"
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Project dir:", PROJECT_DIR)

    cfg = Phase3Config(
        base_model_id="meta-llama/Llama-3.1-8B-Instruct",
        # max_train_samples=30000,
        max_test_samples=None,
        # hard_oversample_ratio=1.5,
        # num_train_epochs=4,
        # per_device_train_batch_size=4,
        # gradient_accumulation_steps=4,
        # learning_rate=2e-4,
        output_base_dir=PROJECT_DIR,
        # output_csv=os.path.join(PROJECT_DIR, "phase2_results.csv"),
        phase3_output_dir=os.path.join(PROJECT_DIR, "phase3_outputs"),
    )

    runner = Phase3Runner(cfg)

    # 这里填入 Phase 2 QLoRA 的 adapter 目录，例如：
    qlora_adapter_dir = os.path.join(PROJECT_DIR, "qlora_r64")  # 根据你 Phase2 的 _output_dir 命名

    runner.run(qlora_adapter_dir=qlora_adapter_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/266_final/phase3
[Dataset] Test: 4941
[Phase3] Loaded dataset - Test: 4941
[Phase3] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Phase3] Model=base | style=base | paraphrase=0


base-0: 100%|██████████| 4941/4941 [12:13<00:00,  6.74it/s]


[Phase3] Model=base | style=base | paraphrase=1


...

[Phase3] Model=qlora | style=cot | paraphrase=1


cot-1: 100%|██████████| 4941/4941 [12:38<00:00,  6.51it/s]


[Phase3] Model=qlora | style=cot | paraphrase=2


cot-2: 100%|██████████| 4941/4941 [12:39<00:00,  6.51it/s]


[Phase3] Saved predictions → /content/drive/MyDrive/266_final/phase3/phase3_outputs/phase3_predictions_qlora.jsonl
[Phase3] Inference done. You can now run separate analysis scripts to compute accuracy, variance, and per-question sensitivity.


In [7]:
import json

path = os.path.join(PROJECT_DIR, "phase3_outputs/phase3_predictions_qlora.jsonl")  # adjust if needed

records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Total records: {len(records)}")
print("First 5 records:")
for r in records[:5]:
    print(r)


Total records: 44469
First 5 records:
{'model': 'qlora', 'index': 0, 'style': 'base', 'paraphrase_id': 0, 'pred': 'D', 'label': 'E', 'question_category': 'Theme Exploration', 'hard_split': True}
{'model': 'qlora', 'index': 1, 'style': 'base', 'paraphrase_id': 0, 'pred': 'E', 'label': 'E', 'question_category': 'Setting and\nTechnical Analysis', 'hard_split': True}
{'model': 'qlora', 'index': 2, 'style': 'base', 'paraphrase_id': 0, 'pred': 'D', 'label': 'D', 'question_category': 'Setting and\nTechnical Analysis', 'hard_split': True}
{'model': 'qlora', 'index': 3, 'style': 'base', 'paraphrase_id': 0, 'pred': 'B', 'label': 'A', 'question_category': 'Setting and\nTechnical Analysis', 'hard_split': False}
{'model': 'qlora', 'index': 4, 'style': 'base', 'paraphrase_id': 0, 'pred': 'D', 'label': 'D', 'question_category': 'Setting and\nTechnical Analysis', 'hard_split': False}


In [8]:
import json
import pandas as pd

# Load jsonl
records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)

# Mark correctness
df["correct"] = df["pred"] == df["label"]

# 1) Overall accuracy (all prompts pooled)
overall_acc = df["correct"].mean()
print(f"Overall accuracy (all prompts pooled): {overall_acc:.3f}")

# 2) Accuracy by question category
cat_stats = df.groupby("question_category")["correct"].mean().reset_index()
cat_stats = cat_stats.rename(columns={"correct": "accuracy"})
print("\nAccuracy by question_category (all prompts pooled):")
print(cat_stats)

# 3) Accuracy by hard_split (True/False)
hard_stats = df.groupby("hard_split")["correct"].mean().reset_index()
hard_stats = hard_stats.rename(columns={"correct": "accuracy"})
print("\nAccuracy by hard_split (all prompts pooled):")
print(hard_stats)


Overall accuracy (all prompts pooled): 0.685

Accuracy by question_category (all prompts pooled):
                      question_category  accuracy
0  Character and\nRelationship Dynamics  0.700570
1          Narrative and\nPlot Analysis  0.696664
2       Setting and\nTechnical Analysis  0.720191
3                              Temporal  0.560562
4                     Theme Exploration  0.649708

Accuracy by hard_split (all prompts pooled):
   hard_split  accuracy
0       False  0.724908
1        True  0.501691


In [ ]:
import os
import json
import math
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm import tqdm


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name_or_path = "meta-llama/Meta-Llama-3-8B-Instruct"  # 举例
qlora_checkpoint_dir = "checkpoints/qlora_cinepile"              # 你自己的路径


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
qlora_model = PeftModel.from_pretrained(qlora_model, qlora_checkpoint_dir)
qlora_model.eval()
base_model.eval()


In [2]:
PROMPT_STYLES = {
    "base": [
        "You are watching a movie. Based on the scene description, answer the multiple-choice question.",
        "Answer the following multiple-choice question about the movie after reviewing the provided scene description.",
        "Given the details of this movie scene, select the correct option for the question provided below."
    ],
    "strict": [
        "You are watching a movie. Based on the scene description, answer the multiple-choice question. Respond with only the letter of the correct option (A, B, C, D, or E).",
        "Examine the movie scene and choose the right answer. Your response must contain nothing but the single capital letter (A-E) representing your choice.",
        "Pick the best option (A, B, C, D, or E) for the movie question below based on the scene. Provide only the letter as your output."
    ],
    "cot": [
        "You are watching a movie. Based on the scene description, think step by step about the question and options, and then answer with the letter of the correct option.",
        "Read the movie scene carefully. Analyze the question and each possible choice logically before providing the final answer letter (A-E).",
        "Evaluate the provided movie scene. First, explain your reasoning process for selecting the best answer, then conclude with the correct letter (A-E)."
    ]
}


In [3]:
def build_prompt(style_name: str, style_prompt: str, scene: str, question: str, options: Dict[str, str]) -> str:
    # options: {"A": "...", "B": "...", ..., "E": "..."}
    options_str = "\n".join([f"{k}. {v}" for k, v in options.items()])
    # 这里使用简单格式，你可以替换成你们正式的 CinePile prompt 格式
    prompt = f"""{style_prompt}

              Scene:
              {scene}

              Question:
              {question}

              Options:
              {options_str}

              Answer:"""
    return prompt


NameError: name 'Dict' is not defined

In [ ]:
@dataclass
class Example:
    scene: str
    question: str
    options: Dict[str, str]  # A-E
    label: str               # correct letter, e.g. "B"
    category: str            # TEMP / CRD / ...

def load_cinepile_test(path: str) -> List[Example]:
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            x = json.loads(line)
            # 你要根据自己的字段名调整
            options = {
                "A": x["option_a"],
                "B": x["option_b"],
                "C": x["option_c"],
                "D": x["option_d"],
                "E": x["option_e"],
            }
            examples.append(
                Example(
                    scene=x["scene"],
                    question=x["question"],
                    options=options,
                    label=x["answer"],      # "A"~"E"
                    category=x["category"]  # TEMP / ...
                )
            )
    return examples

test_examples = load_cinepile_test("data/cinepile_test.jsonl")


In [ ]:
import re

CHOICE_LETTERS = ["A", "B", "C", "D", "E"]

def extract_choice_from_output(text: str) -> str:
    # 大写化、去空格影响
    t = text.strip()
    # 优先匹配类似 "Answer: B" / "(B)" / "B."
    pattern = r"[A-E]"
    m = re.search(pattern, t)
    if m:
        return m.group(0)
    return None  # 表示解析失败，可以单独统计


In [ ]:
@torch.inference_mode()
def run_inference_single_prompt(
    model,
    tokenizer,
    examples: List[Example],
    style_name: str,
    style_prompt: str,
    max_new_tokens: int = 64,
    batch_size: int = 8
):
    """
    返回一个列表，每个元素是 {
        "pred": "A"/.../None,
        "gold": "B",
        "category": "TEMP",
        "scene_id": i  # 或者 index
    }
    """
    results = []

    for i in tqdm(range(0, len(examples), batch_size), desc=f"{style_name} / paraphrase"):
        batch = examples[i:i+batch_size]
        prompts = [
            build_prompt(style_name, style_prompt, ex.scene, ex.question, ex.options)
            for ex in batch
        ]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
        outputs = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        # 只取新生成部分
        generated = outputs[:, enc["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for j, ex in enumerate(batch):
            pred_choice = extract_choice_from_output(texts[j])
            results.append({
                "index": i + j,
                "pred": pred_choice,
                "gold": ex.label,
                "category": ex.category,
                "style": style_name,
                "prompt_text": style_prompt  # 可选：只保存标识
            })
    return results


In [ ]:
def run_all_prompts_for_model(model, model_name: str, examples: List[Example], out_path: str):
    all_results = []
    for style_name, paraphrases in PROMPT_STYLES.items():
        for pid, style_prompt in enumerate(paraphrases):
            print(f"Running {model_name} | style={style_name} | paraphrase {pid}")
            res = run_inference_single_prompt(
                model=model,
                tokenizer=tokenizer,
                examples=examples,
                style_name=style_name,
                style_prompt=style_prompt,
                max_new_tokens=64,
                batch_size=8
            )
            # 加上 paraphrase id
            for r in res:
                r["paraphrase_id"] = pid
                r["model"] = model_name
            all_results.extend(res)

    with open(out_path, "w", encoding="utf-8") as f:
        for r in all_results:
            f.write(json.dumps(r) + "\n")
    print(f"Saved {len(all_results)} predictions to {out_path}")


In [ ]:
run_all_prompts_for_model(
    model=base_model,
    model_name="base",
    examples=test_examples,
    out_path="results/base_phase3_predictions.jsonl"
)

run_all_prompts_for_model(
    model=qlora_model,
    model_name="qlora",
    examples=test_examples,
    out_path="results/qlora_phase3_predictions.jsonl"
)


In [ ]:
def load_predictions(path: str) -> List[Dict[str, Any]]:
    preds = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            preds.append(json.loads(line))
    return preds

base_preds = load_predictions("results/base_phase3_predictions.jsonl")
qlora_preds = load_predictions("results/qlora_phase3_predictions.jsonl")
all_preds = base_preds + qlora_preds


In [ ]:
from collections import defaultdict

def compute_accuracy(preds: List[Dict[str, Any]]) -> float:
    total = 0
    correct = 0
    for r in preds:
        if r["pred"] is None:
            continue  # 或者算错；你可以单独记一个 parse_error_rate
        total += 1
        if r["pred"] == r["gold"]:
            correct += 1
    return correct / total if total > 0 else 0.0


In [ ]:
def eval_accuracy_by_style_and_category(preds: List[Dict[str, Any]]):
    grouped = defaultdict(list)
    # key: (model, style, paraphrase_id, category)
    for r in preds:
        key = (r["model"], r["style"], r["paraphrase_id"], r["category"])
        grouped[key].append(r)

    # 先算 paraphrase 粒度
    stats = []
    for (model, style, pid, cat), group in grouped.items():
        acc = compute_accuracy(group)
        stats.append({
            "model": model,
            "style": style,
            "paraphrase_id": pid,
            "category": cat,
            "acc": acc
        })
    return stats

stats = eval_accuracy_by_style_and_category(all_preds)


In [ ]:
import statistics

def compute_style_paraphrase_variance(stats):
    # stats: list of dict(model, style, paraphrase_id, category, acc)
    style_group = defaultdict(list)
    for s in stats:
        key = (s["model"], s["style"], s["category"])
        style_group[key].append(s["acc"])

    style_stats = []
    for (model, style, cat), accs in style_group.items():
        mean_acc = sum(accs) / len(accs)
        std_acc = statistics.pstdev(accs) if len(accs) > 1 else 0.0
        style_stats.append({
            "model": model,
            "style": style,
            "category": cat,
            "mean_acc": mean_acc,
            "std_acc": std_acc
        })
    return style_stats

style_stats = compute_style_paraphrase_variance(stats)


In [ ]:
def compute_question_sensitivity(preds: List[Dict[str, Any]]):
    # group by (model, index)
    q_group = defaultdict(list)
    for r in preds:
        key = (r["model"], r["index"])
        q_group[key].append(r)

    q_stats = []
    for (model, idx), group in q_group.items():
        preds_list = [g["pred"] for g in group if g["pred"] is not None]
        gold = group[0]["gold"]
        category = group[0]["category"]

        if len(preds_list) == 0:
            unique_answers = 0
            sensitivity = 1.0  # 全部解析失败，可以特殊处理
        else:
            unique_answers = len(set(preds_list))
            # 1 - max_freq / N, 越接近 0 越稳定
            from collections import Counter
            c = Counter(preds_list)
            max_freq = max(c.values())
            sensitivity = 1.0 - max_freq / len(preds_list)

        q_stats.append({
            "model": model,
            "index": idx,
            "category": category,
            "gold": gold,
            "num_preds": len(group),
            "num_unique_answers": unique_answers,
            "sensitivity": sensitivity
        })
    return q_stats

q_stats = compute_question_sensitivity(all_preds)


In [ ]:
def summarize_sensitivity(q_stats):
    agg = defaultdict(list)
    for q in q_stats:
        key_all = (q["model"], "ALL")
        key_cat = (q["model"], q["category"])
        agg[key_all].append(q["sensitivity"])
        agg[key_cat].append(q["sensitivity"])

    summary = []
    for (model, cat), vals in agg.items():
        mean_sens = sum(vals) / len(vals)
        summary.append({
            "model": model,
            "category": cat,
            "mean_sensitivity": mean_sens,
            "num_questions": len(vals)
        })
    return summary

sens_summary = summarize_sensitivity(q_stats)
